# HDT 1: Pandas, SQL y DuckDB

**Ciencia de Datos, Sección A** · Asignada: martes 28 de julio · **Entrega: martes 4 de agosto, 23:59**

**Nombre:** _(escribir aquí)_

Completar las celdas marcadas con `# ¿Qué va aquí?`. Cada ejercicio incluye una verificación comentada: descomentar para comprobar el resultado. Antes de entregar: **Kernel → Restart & Run All** (un notebook que no corre de arriba a abajo pierde 0.5 pts).

AI: resolver sin AI. Si se usó para entender un concepto, anotarlo en la mini-bitácora del final.

## Setup

Si falta DuckDB: descomentar la línea de instalación, ejecutar la celda una vez y volver a comentarla.

In [ ]:
# !uv add duckdb    (en terminal)  o descomentar:  %pip install duckdb
import pandas as pd
import duckdb

URL = ("https://raw.githubusercontent.com/"
       "mwaskom/seaborn-data/master/penguins.csv")
penguins = pd.read_csv(URL)

# Tabla de nombres científicos (para los JOIN)
especies = pd.DataFrame({
    "species": ["Adelie", "Chinstrap", "Gentoo"],
    "nombre_cientifico": ["Pygoscelis adeliae",
                          "Pygoscelis antarcticus",
                          "Pygoscelis papua"],
})
penguins.head()

## Parte A · Pandas (1.0 pt)

### Ejercicio 1 (0.10): cargar y explorar

Mostrar: (a) el `shape` del DataFrame, (b) los `dtypes`, y (c) cuántos nulos tiene **cada columna**.

In [ ]:
# ¿Que va aqui? (tres expresiones, una por inciso)
# (a) dimensiones  -> atributo .shape  (tupla filas, columnas)
# (b) tipos        -> atributo .dtypes
# (c) nulos por columna -> .isna() devuelve booleanos; sumar por columna con .sum()

print("shape:", ...)
print(...)
print(...)

# Verificacion: el dataset original tiene 344 filas y 7 columnas,
# y la columna sex es la que mas nulos tiene (11).

### Ejercicio 2 (0.15): limpieza mínima

Crear un DataFrame `limpio` **sin** las filas que tengan nulo en cualquier columna. Reportar con un `print` cuántas filas se perdieron respecto al original.

In [ ]:
# Pista: el metodo .dropna() elimina filas con NA.
#        Por defecto how="any" (basta un NA en la fila).
limpio = ...   # ¿Que va aqui?

# filas perdidas = filas del original - filas de limpio  (usar .shape[0])
# print(f"Se perdieron {...} filas")

# Verificacion (descomentar):
# assert limpio.shape[0] == 333 and limpio.isna().sum().sum() == 0

### Ejercicio 3 (0.15): máscaras y orden

De `limpio`: los pingüinos de la isla **Biscoe** con masa corporal **mayor a 4500 g**, ordenados de mayor a menor masa. Mostrar solo las columnas `species`, `island`, `body_mass_g`.

In [ ]:
# Receta de mascara booleana:
#   m = (limpio["island"] == ...) & (limpio["body_mass_g"] > ...)
#   limpio.loc[m, [columnas]].sort_values(por, ascending=False)
# Ojo: cada condicion va entre parentesis, y el "y" logico es &, no "and".

cols = ["species", "island", "body_mass_g"]
mascara = ...          # ¿Que va aqui?
pesados_biscoe = ...   # ¿Que va aqui?
pesados_biscoe

# Verificacion (descomentar):
# assert (pesados_biscoe["island"] == "Biscoe").all()
# assert (pesados_biscoe["body_mass_g"] > 4500).all()
# assert pesados_biscoe["body_mass_g"].is_monotonic_decreasing

### Ejercicio 4 (0.20): groupby con dos funciones

Masa corporal por **especie y sexo**: el **promedio** y el **conteo**, en una sola operación con `groupby` + `agg`.

In [ ]:
# Receta: df.groupby([clave1, clave2])[columna].agg([func1, func2])
#   - las claves van en lista si son dos
#   - "mean" y "count" se pueden pasar como strings
resumen = ...   # ¿Que va aqui?
resumen

# Verificacion: el grupo mas pesado debe ser Gentoo macho (~5485 g en promedio).

### Ejercicio 5 (0.20): columna derivada

Agregar a `limpio` una columna `bill_ratio` = largo del pico / profundidad del pico. Mostrar el promedio de `bill_ratio` **por especie**, ordenado descendente. ¿Qué especie tiene el pico proporcionalmente más alargado?

In [ ]:
# (a) columna nueva: limpio["bill_ratio"] = limpio[largo] / limpio[profundidad]
#     (las columnas son bill_length_mm y bill_depth_mm)
# (b) promedio por especie: groupby("species")["bill_ratio"].mean()
#     y luego .sort_values(ascending=False)

limpio["bill_ratio"] = ...   # ¿Que va aqui?
por_especie = ...            # ¿Que va aqui?
por_especie

# Verificacion: Gentoo debe quedar de primero (~3.2).

### Ejercicio 6 (0.20): merge

Unir `limpio` con la tabla `especies` para que cada fila tenga su `nombre_cientifico`. Mostrar una fila de cada especie para comprobar.

In [ ]:
# Receta: pd.merge(izquierda, derecha, on=<columna comun>, how="left")
#   how="left" conserva todas las filas de limpio (por eso el assert de shape)
con_nombres = ...   # ¿Que va aqui?

# con_nombres.drop_duplicates("species")[["species", "nombre_cientifico"]]

# Verificacion (descomentar):
# assert con_nombres.shape[0] == limpio.shape[0]
# assert "nombre_cientifico" in con_nombres.columns

## Parte B · SQL con DuckDB (0.8 pt)

DuckDB consulta directamente los DataFrames en memoria: `duckdb.sql("SELECT ... FROM limpio")`. Cerrar cada consulta con `.df()` para ver el resultado como DataFrame.

### Ejercicio 7 (0.20): SELECT / WHERE / ORDER BY

El ejercicio 3, ahora en SQL: especie, isla y masa de los pingüinos de Biscoe con masa mayor a 4500 g, ordenados de mayor a menor.

In [ ]:
q7 = """
-- ¿Que va aqui?
-- SELECT   <columnas>
-- FROM     limpio          -- DuckDB lee el DataFrame de Python por su nombre
-- WHERE    <isla> AND <masa>
-- ORDER BY <columna> DESC
"""
duckdb.sql(q7).df()

# Verificacion: debe dar las mismas filas que el ejercicio 3.

### Ejercicio 8 (0.20): GROUP BY + HAVING

Especies cuya masa corporal **promedio** supera los 4000 g, con su promedio redondeado.

In [ ]:
q8 = """
-- ¿Que va aqui?
-- SELECT   species, ROUND(AVG(<columna>), <decimales>) AS masa_prom
-- FROM     limpio
-- GROUP BY <columna de agrupacion>
-- HAVING   <condicion sobre el agregado>   -- HAVING filtra despues de agrupar
"""
duckdb.sql(q8).df()

# Verificacion: solo una especie debe aparecer. ¿Cual? Comparar con el resultado
# del ejercicio 4.

### Ejercicio 9 (0.20): JOIN

El ejercicio 6, ahora en SQL: unir `limpio` con `especies` y mostrar especie, nombre científico y masa promedio por especie.

In [ ]:
q9 = """
-- ¿Que va aqui?
-- SELECT   l.species, e.nombre_cientifico, AVG(...) AS masa_prom
-- FROM     limpio l
-- JOIN     especies e ON l.<col> = e.<col>
-- GROUP BY <todas las columnas no agregadas>
"""
duckdb.sql(q9).df()

# Verificacion: 3 filas, una por especie, cada una con su Pygoscelis.

### Ejercicio 10 (0.20): window function

Los **3 pingüinos más pesados de cada especie**, usando `RANK() OVER (PARTITION BY ... ORDER BY ...)`. Pista de la sesión 3: la window function se calcula en una subconsulta y se filtra afuera.

In [ ]:
q10 = """
-- ¿Que va aqui?  Patron: window en subconsulta, filtro afuera.
-- SELECT species, body_mass_g, rk
-- FROM (
--     SELECT species,
--            body_mass_g,
--            RANK() OVER (PARTITION BY <columna> ORDER BY <columna> DESC) AS rk
--     FROM limpio
-- )
-- WHERE rk <= <n>
-- ORDER BY species, rk
"""
duckdb.sql(q10).df()

# Verificacion: alrededor de 9 filas (3 por especie; puede haber empates),
# y el rango nunca debe pasar de 3.

## Parte C · Criterio (0.2 pt)

### Ejercicio 11 (0.20)

Los mismos análisis se resolvieron en Pandas y en SQL. En 3-4 líneas, **con base en el trabajo de esta hoja** (no de memoria): ¿cuándo conviene cada herramienta? Mencionar al menos una operación que resultó más natural en cada una.

_(Responder editando esta celda)_

**Respuesta:** ...

## Mini-bitácora de AI (opcional, no penaliza)

Si se usó AI para entender algún concepto, anotar aquí qué se preguntó y qué se entendió. Si no se usó, escribir "No se usó".

- ...

## Anexo: repaso de las sesiones 2 y 3

Regla de los ejercicios de NumPy: **sin ciclos `for`**.

In [ ]:
import numpy as np

rng = np.random.default_rng(7)

### A1 (sesión 2): z-score sin loops

Normalizar un array: restar la media y dividir entre la desviación estándar.

In [ ]:
alturas = rng.normal(170, 10, size=1000)

def z_score(x):
    # ¿Que va aqui? (sin for)
    # z = (cada valor - la media del array) / la desviacion estandar
    # x.mean() y x.std() operan sobre todo el array; la resta y la division
    # se aplican elemento por elemento (vectorizacion).
    return ...

z = z_score(alturas)
# Verificacion (descomentar):
# print(round(z.mean(), 4), round(z.std(), 4))  # ~0 y ~1

### A2 (sesión 2): distancias con broadcasting

Distancia euclidiana de cada punto a un centro, sin loops.

In [ ]:
puntos = rng.normal(size=(500, 2))   # 500 puntos en 2D
centro = np.array([1.0, 1.0])

# ¿Que va aqui?
# Pista: (puntos - centro) usa broadcasting (500,2) - (2,)
# Luego: elevar al cuadrado, sumar con axis=1, sacar raiz
#   diferencias = puntos - centro        -> (500, 2)
#   distancias  = np.sqrt((diferencias ** 2).sum(axis=...))
diferencias = ...
distancias = ...

# ¿Cuantos puntos estan a menos de 1 del centro?
# Pista: (distancias < 1) da booleanos; sumarlos cuenta los True.
cercanos = ...

# Verificacion (descomentar):
# print(distancias.shape)  # (500,)
# print(cercanos)

### A3 (sesión 3): propinas por día y turno

Dataset `tips` (propinas de un restaurante). `pct` = propina como fracción de la cuenta.

In [ ]:
URL_TIPS = ("https://raw.githubusercontent.com/"
            "mwaskom/seaborn-data/master/tips.csv")
tips = pd.read_csv(URL_TIPS)
tips["pct"] = tips["tip"] / tips["total_bill"]
tips.head()

In [ ]:
# a) porcentaje medio de propina por dia
# b) por dia Y turno (day, time), en una tabla
# c) el dia con el mayor porcentaje medio
# d) numero de mesas por dia

# Receta: tips.groupby("day").agg(
#             pct_medio=("pct", "mean"),
#             mesas=("pct", "count"),
#         )
#   -> named aggregation: nombre_nuevo=(columna, funcion)
# Para (b): agrupar por ["day", "time"] en otra celda o variable aparte.
# Para (c): sobre el resumen, .idxmax() de la columna del promedio.

resumen = ...  # ¿Que va aqui? (usar agg)

# Verificacion: resumen debe tener 4 filas
# print(resumen.shape)